# 勇者傳說 — Building Generator (T4 GPU)

Generate **AI pixel art buildings** for all 12 kingdoms using SDXL + pixel-art-xl LoRA.

| Type | Count | Generate Size | Final Size | Steps | Guidance |
|------|-------|--------------|------------|-------|----------|
| Castles | 12 | 512×512 | 320×320 | 25 | 8.0 |
| Gates | 12 | 384×256 | 192×128 | 25 | 7.5 |
| Inns | 12 | 256×256 | 128×128 | 25 | 7.5 |
| Shops | 12 | 256×256 | 128×128 | 25 | 7.5 |
| Churches | 12 | 256×256 | 128×128 | 25 | 7.5 |
| Houses | 12 | 256×256 | 128×128 | 25 | 7.5 |
| **Total** | **72** | | | | |

**Self-contained**: No repo clone needed — all code + prompts embedded.  
**Output**: Google Drive `/MyDrive/ai-rpg-game/outputs/`  
**Resume**: 中斷後重跑自動跳過已完成的

In [2]:
!pip install -q "numpy<2" scipy
!pip install -q diffusers transformers accelerate safetensors "Pillow>=10,<12" rembg onnxruntime

import gc, json, os, time
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

GDRIVE_BASE = Path('/content/drive/MyDrive/ai-rpg-game')
OUTPUT_BASE = GDRIVE_BASE / 'outputs' if ON_COLAB else Path('outputs')
MODELS_DIR = GDRIVE_BASE / 'models' if ON_COLAB else Path('models')

for d in [OUTPUT_BASE, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1024**3
    DEVICE = 'cuda'
    DTYPE = torch.float16
    print(f'GPU: {name} ({vram:.1f} GB VRAM)')
else:
    DEVICE = 'cpu'
    DTYPE = torch.bfloat16
    print('WARNING: No GPU! Generation will be very slow.')

print(f'Output: {OUTPUT_BASE}')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0

In [3]:
from PIL import Image
import numpy as np


class ProgressTracker:
    def __init__(self, task_name, output_dir):
        self.file = Path(output_dir) / f'_progress_{task_name}.json'
        self.completed = set()
        self.start_time = time.time()
        if self.file.exists():
            try:
                data = json.loads(self.file.read_text())
                self.completed = set(data.get('completed', []))
                print(f'[resume] {len(self.completed)} items already done')
            except Exception:
                pass

    def is_done(self, name): return name in self.completed

    def mark_done(self, name):
        self.completed.add(name)
        self.file.parent.mkdir(parents=True, exist_ok=True)
        self.file.write_text(json.dumps({
            'completed': sorted(self.completed),
            'count': len(self.completed),
            'last_updated': time.strftime('%Y-%m-%d %H:%M:%S'),
        }, indent=2))

    def summary(self, total):
        done = len(self.completed)
        elapsed = time.time() - self.start_time
        if done > 0:
            remaining = (total - done) * (elapsed / done)
            return f'{done}/{total} done, ~{remaining/60:.0f} min remaining'
        return f'0/{total} done'


def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def print_vram():
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated(0) / 1024**3
        props = torch.cuda.get_device_properties(0)
        total = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1024**3
        print(f'[vram] {used:.1f}/{total:.1f} GB')


_rmbg_model = None
_rembg_session = None


def remove_background(img):
    """Remove background: RMBG-2.0 primary, rembg fallback."""
    global _rmbg_model, _rembg_session
    img_rgba = img.convert('RGBA')

    # Try RMBG-2.0 first
    try:
        if _rmbg_model is None:
            from transformers import pipeline as hf_pipeline
            _rmbg_model = hf_pipeline("image-segmentation", model="briaai/RMBG-2.0",
                                       trust_remote_code=True, device=0 if torch.cuda.is_available() else -1)
        result = _rmbg_model(img.convert('RGB'))
        mask = result[0]['mask'].convert('L')
        img_rgba.putalpha(mask)
        return img_rgba
    except Exception as e:
        print(f'[warn] RMBG-2.0 failed ({e}), trying rembg...')

    # Fallback: rembg u2net
    try:
        from rembg import remove, new_session
        if _rembg_session is None:
            _rembg_session = new_session('u2net')
        return remove(img_rgba, session=_rembg_session)
    except Exception as e:
        print(f'[warn] rembg also failed: {e}')
        return img_rgba


def quality_check(img, min_opaque_pct=0.10, min_bbox_pct=0.20):
    """Check image quality: enough opaque pixels and content coverage."""
    arr = np.array(img)
    if arr.shape[2] < 4:
        return True  # No alpha = assume OK
    alpha = arr[:,:,3]
    total = alpha.size
    opaque = np.sum(alpha > 10)
    opaque_pct = opaque / total

    if opaque_pct < min_opaque_pct:
        return False

    rows = np.any(alpha > 10, axis=1)
    cols = np.any(alpha > 10, axis=0)
    if not rows.any():
        return False
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    bbox_area = (rmax - rmin + 1) * (cmax - cmin + 1)
    bbox_pct = bbox_area / total

    return bbox_pct >= min_bbox_pct

In [4]:
REGIONS = [
    {'id': 'r1', 'name': 'region_hero', 'theme': 'classic medieval European kingdom', 'elements': 'stone and timber walls, blue banners, peaked slate roofs, iron-bound doors'},
    {'id': 'r2', 'name': 'region_elf', 'theme': 'elven forest kingdom', 'elements': 'living wood structure, golden filigree, green vine decorations, leaf-shaped windows'},
    {'id': 'r3', 'name': 'region_treant', 'theme': 'ancient treehouse village', 'elements': 'hollow tree trunk walls, moss covering, mushroom platforms, root foundations'},
    {'id': 'r4', 'name': 'region_beast', 'theme': 'tribal savanna settlement', 'elements': 'mud-brick and clay walls, animal pelt awnings, bone decorations, thatched roofs'},
    {'id': 'r5', 'name': 'region_merfolk', 'theme': 'coral aquatic kingdom', 'elements': 'shell and coral architecture, blue-white palette, pearl accents, wave patterns'},
    {'id': 'r6', 'name': 'region_giant', 'theme': 'massive stone fortress', 'elements': 'oversized proportions, grey granite blocks, iron rivets, heavy stone archways'},
    {'id': 'r7', 'name': 'region_dwarf', 'theme': 'underground forge city', 'elements': 'carved stone facade, forge glow from windows, copper and bronze trim, anvil motifs'},
    {'id': 'r8', 'name': 'region_undead', 'theme': 'gothic dark castle ruins', 'elements': 'purple glow, cobwebs, gargoyles, cracked dark stone, eerie green lights'},
    {'id': 'r9', 'name': 'region_volcano', 'theme': 'obsidian volcanic tribal', 'elements': 'basalt walls, lava glow cracks, fire motifs, dark stone with ember accents'},
    {'id': 'r10', 'name': 'region_hotspring', 'theme': 'Japanese onsen town', 'elements': 'wooden beams, bamboo accents, paper lanterns, sliding doors, curved roofs'},
    {'id': 'r11', 'name': 'region_mountain', 'theme': 'alpine mountain lodge', 'elements': 'snow-capped roof, prayer flags, heavy timber, stone chimney, warm glow'},
    {'id': 'r12', 'name': 'region_demon', 'theme': 'dark demonic fortress', 'elements': 'bone and skull motifs, red banners, demonic carvings, dark iron spikes'},
]

BUILDING_TYPES = [
    {
        'type': 'castle', 'prefix': 'bld_castle',
        'desc': 'grand royal castle with towers and main gate, imposing multi-story palace',
        'gen_w': 512, 'gen_h': 512, 'final_w': 320, 'final_h': 320,
        'steps': 25, 'guidance': 8.0, 'lora_weight': 0.6,
    },
    {
        'type': 'gate', 'prefix': 'bld_gate',
        'desc': 'fortified entrance gate archway with portcullis, wide stone gatehouse',
        'gen_w': 384, 'gen_h': 256, 'final_w': 192, 'final_h': 128,
        'steps': 25, 'guidance': 7.5, 'lora_weight': 0.8,
    },
    {
        'type': 'inn', 'prefix': 'bld_inn',
        'desc': 'cozy inn tavern building with hanging lantern sign, warm light from windows',
        'gen_w': 256, 'gen_h': 256, 'final_w': 128, 'final_h': 128,
        'steps': 25, 'guidance': 7.5, 'lora_weight': 0.8,
    },
    {
        'type': 'shop', 'prefix': 'bld_shop',
        'desc': 'merchant shop building with display window and goods, shop sign',
        'gen_w': 256, 'gen_h': 256, 'final_w': 128, 'final_h': 128,
        'steps': 25, 'guidance': 7.5, 'lora_weight': 0.8,
    },
    {
        'type': 'church', 'prefix': 'bld_church',
        'desc': 'sacred church temple building with steeple cross bell tower, stained glass window',
        'gen_w': 256, 'gen_h': 256, 'final_w': 128, 'final_h': 128,
        'steps': 25, 'guidance': 7.5, 'lora_weight': 0.8,
    },
    {
        'type': 'house', 'prefix': 'bld_region',
        'desc': 'residential house dwelling, simple home with door and windows',
        'gen_w': 256, 'gen_h': 256, 'final_w': 128, 'final_h': 128,
        'steps': 25, 'guidance': 7.5, 'lora_weight': 0.8,
    },
]

# Build flat list of all 72 prompts
PROMPTS = {
    '_meta': {
        'style_prefix_xl': '16-bit JRPG pixel art, top-down RPG building sprite',
        'style_suffix_building_xl': 'front view, single building centered, transparent background, clean edges, detailed pixel art',
        'negative_prompt_xl': 'blurry, modern, multiple buildings, text, UI, photograph, realistic photo, people, characters, watermark',
    },
    'buildings': [],
}

for region in REGIONS:
    for btype in BUILDING_TYPES:
        name = f"{btype['prefix']}_{region['id']}"
        prompt_body = f"{region['theme']} {btype['desc']}, {region['elements']}"
        PROMPTS['buildings'].append({
            'name': name,
            'prompt_xl': prompt_body,
            'gen_w': btype['gen_w'],
            'gen_h': btype['gen_h'],
            'final_w': btype['final_w'],
            'final_h': btype['final_h'],
            'steps': btype['steps'],
            'guidance': btype['guidance'],
            'lora_weight': btype['lora_weight'],
            'type': btype['type'],
            'region': region['name'],
        })

print(f"Total prompts: {len(PROMPTS['buildings'])}")
for btype in BUILDING_TYPES:
    count = sum(1 for p in PROMPTS['buildings'] if p['type'] == btype['type'])
    print(f"  {btype['type']}: {count}")

Total prompts: 72
  castle: 12
  gate: 12
  inn: 12
  shop: 12
  church: 12
  house: 12


In [5]:
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

SDXL_HUB = 'stabilityai/stable-diffusion-xl-base-1.0'
LORA_HUB = 'nerijs/pixel-art-xl'

sdxl_cache = MODELS_DIR / 'stable-diffusion-xl-base-1.0'
model_path = str(sdxl_cache) if (sdxl_cache / 'model_index.json').exists() else SDXL_HUB

print(f'[pipeline] Loading SDXL ({DEVICE})...')
t0 = time.time()

kwargs = {'torch_dtype': DTYPE, 'use_safetensors': True, 'low_cpu_mem_usage': False}
if 'stabilityai' in model_path:
    kwargs['variant'] = 'fp16'

pipe = StableDiffusionXLPipeline.from_pretrained(model_path, **kwargs)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type='dpmsolver++',
    use_karras_sigmas=True,
)

try:
    pipe.load_lora_weights(LORA_HUB, adapter_name='pixel_xl')
    pipe.set_adapters(['pixel_xl'], adapter_weights=[0.8])
    print('[pipeline] pixel-art-xl LoRA loaded (weight=0.8)')
except Exception as e:
    print(f'[warn] LoRA failed: {e}')

pipe = pipe.to(DEVICE)
pipe.enable_attention_slicing()

# Cache to Drive for future runs
if 'stabilityai' in model_path and ON_COLAB:
    try:
        pipe.save_pretrained(str(sdxl_cache))
        print(f'[pipeline] Cached to {sdxl_cache}')
    except Exception:
        pass

print(f'[pipeline] Ready in {time.time()-t0:.1f}s')
print_vram()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


[pipeline] Loading SDXL (cuda)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

...

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModelWithProjection related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


[pipeline] pixel-art-xl LoRA loaded (weight=0.8)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[pipeline] Cached to /content/drive/MyDrive/ai-rpg-game/models/stable-diffusion-xl-base-1.0
[pipeline] Ready in 246.2s
[vram] 6.7/14.6 GB


In [6]:
def postprocess_building(img, entry):
    """Remove background, resize to final building size."""
    final_w = entry['final_w']
    final_h = entry['final_h']

    # Remove background
    img = remove_background(img.convert('RGBA'))

    # Crop to content bounding box
    arr = np.array(img)
    alpha = arr[:, :, 3]
    rows = np.any(alpha > 10, axis=1)
    cols = np.any(alpha > 10, axis=0)
    if not rows.any():
        return img.resize((final_w, final_h), Image.NEAREST)

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    cropped = img.crop((cmin, rmin, cmax + 1, rmax + 1))

    # Scale to fit final size (maintain aspect ratio)
    cw, ch = cropped.size
    scale = min(final_w / cw, final_h / ch)
    new_w = max(1, int(cw * scale))
    new_h = max(1, int(ch * scale))
    resized = cropped.resize((new_w, new_h), Image.NEAREST)

    # Center horizontally, bottom-align vertically on transparent canvas
    canvas = Image.new('RGBA', (final_w, final_h), (0, 0, 0, 0))
    x = (final_w - new_w) // 2
    y = final_h - new_h
    canvas.paste(resized, (x, y))
    return canvas


def generate_building(entry, meta, out_dir, tracker, max_retries=3):
    """Generate a single building sprite with quality gate and retries."""
    name = entry['name']
    out_path = out_dir / f'{name}.png'

    if tracker.is_done(name) and out_path.exists():
        print(f'  [skip] {name}')
        return

    prefix = meta['style_prefix_xl']
    suffix = meta['style_suffix_building_xl']
    neg = meta['negative_prompt_xl']
    prompt = f"{prefix}, {entry['prompt_xl']}, {suffix}"

    steps = entry['steps']
    guidance = entry['guidance']
    gen_w = entry['gen_w']
    gen_h = entry['gen_h']
    lora_weight = entry.get('lora_weight', 0.8)

    # Adjust LoRA weight per building type
    pipe.set_adapters(['pixel_xl'], adapter_weights=[lora_weight])

    print(f"  {name} ({entry['type']}, {gen_w}x{gen_h}\u2192{entry['final_w']}x{entry['final_h']})...", end=' ')
    t0 = time.time()

    for attempt in range(max_retries):
        seed = 42 + attempt * 1000
        generator = torch.Generator(device=DEVICE).manual_seed(seed)

        result = pipe(
            prompt=prompt, negative_prompt=neg,
            width=gen_w, height=gen_h,
            num_inference_steps=steps,
            guidance_scale=guidance,
            generator=generator,
        )
        img = postprocess_building(result.images[0], entry)

        if quality_check(img):
            img.save(str(out_path), 'PNG')
            kb = out_path.stat().st_size / 1024
            print(f'{kb:.0f} KB, {time.time()-t0:.1f}s' + (f' (attempt {attempt+1})' if attempt > 0 else ''))
            tracker.mark_done(name)
            free_vram()
            return
        else:
            print(f'[retry {attempt+1}]', end=' ')

    # Save last attempt anyway
    img.save(str(out_path), 'PNG')
    kb = out_path.stat().st_size / 1024
    print(f'{kb:.0f} KB, {time.time()-t0:.1f}s [quality warning]')
    tracker.mark_done(name)
    free_vram()

In [7]:
entries = PROMPTS['buildings']
meta = PROMPTS['_meta']
out_dir = OUTPUT_BASE / 'buildings'
out_dir.mkdir(parents=True, exist_ok=True)

tracker = ProgressTracker('buildings', OUTPUT_BASE)
remaining = sum(1 for e in entries if not tracker.is_done(e['name']))
print(f'\nBuildings: {len(entries)} total, {remaining} to generate')

# Group by type for organized output
for btype in BUILDING_TYPES:
    type_entries = [e for e in entries if e['type'] == btype['type']]
    type_remaining = sum(1 for e in type_entries if not tracker.is_done(e['name']))
    if type_remaining == 0:
        print(f'\n=== {btype["type"]} === (all done)')
        continue

    print(f'\n=== {btype["type"]} ({type_remaining} remaining) ===')
    for i, entry in enumerate(type_entries):
        print(f'[{i+1}/{len(type_entries)}] {tracker.summary(len(entries))}')
        generate_building(entry, meta, out_dir, tracker)

print(f'\nAll buildings done!')


Buildings: 72 total, 72 to generate

=== castle (12 remaining) ===
[1/12] 0/72 done
  bld_castle_r1 (castle, 512x512→320x320)... 

  0%|          | 0/25 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


[warn] RMBG-2.0 failed (You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/briaai/RMBG-2.0.
401 Client Error. (Request ID: Root=1-69ae2c56-45fbb7c013209d896014a865;2fe443d1-676a-4174-bcbc-b543be2e5ba1)

Cannot access gated repo for url https://huggingface.co/briaai/RMBG-2.0/resolve/main/config.json.
Access to model briaai/RMBG-2.0 is restricted. You must have access to it and be authenticated to access it. Please log in.), trying rembg...


  0%|                                               | 0.00/176M [00:00<?, ?B/s]

...

[warn] RMBG-2.0 failed (You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/briaai/RMBG-2.0.
401 Client Error. (Request ID: Root=1-69ae2fb9-0d9d00de556c8a600281b58c;5602c629-f7a9-4689-851c-f874a166cd3d)

Cannot access gated repo for url https://huggingface.co/briaai/RMBG-2.0/resolve/main/config.json.
Access to model briaai/RMBG-2.0 is restricted. You must have access to it and be authenticated to access it. Please log in.), trying rembg...
20 KB, 10.4s
[11/12] 70/72 done, ~0 min remaining
  bld_region_r11 (house, 256x256→128x128)... 

  0%|          | 0/25 [00:00<?, ?it/s]

[warn] RMBG-2.0 failed (You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/briaai/RMBG-2.0.
401 Client Error. (Request ID: Root=1-69ae2fc3-3a961a991cd3d2234989af74;0217fe84-713b-4b6a-b11b-0e04ecdfe920)

Cannot access gated repo for url https://huggingface.co/briaai/RMBG-2.0/resolve/main/config.json.
Access to model briaai/RMBG-2.0 is restricted. You must have access to it and be authenticated to access it. Please log in.), trying rembg...
18 KB, 10.6s
[12/12] 71/72 done, ~0 min remaining
  bld_region_r12 (house, 256x256→128x128)... 

  0%|          | 0/25 [00:00<?, ?it/s]

[warn] RMBG-2.0 failed (You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/briaai/RMBG-2.0.
401 Client Error. (Request ID: Root=1-69ae2fcf-78b1300d499a03d559379ab6;e6fd9446-2ebf-4469-b103-5e0216989488)

Cannot access gated repo for url https://huggingface.co/briaai/RMBG-2.0/resolve/main/config.json.
Access to model briaai/RMBG-2.0 is restricted. You must have access to it and be authenticated to access it. Please log in.), trying rembg...
23 KB, 9.9s

All buildings done!


In [8]:
import matplotlib.pyplot as plt

out_dir = OUTPUT_BASE / 'buildings'
pngs = sorted(out_dir.glob('*.png'))
pngs = [p for p in pngs if not p.stem.startswith('_')]

# Group by type for organized preview
type_order = ['castle', 'gate', 'inn', 'shop', 'church', 'region']
for btype in type_order:
    type_pngs = [p for p in pngs if p.stem.startswith(f'bld_{btype}_')]
    if not type_pngs:
        continue

    cols = min(6, len(type_pngs))
    rows = (len(type_pngs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)
    for ax in axes.flat:
        ax.axis('off')

    for i, png in enumerate(type_pngs):
        ax = axes[i // cols][i % cols]
        img = Image.open(png)
        ax.imshow(img)
        label = png.stem.replace(f'bld_{btype}_', '')
        ax.set_title(label, fontsize=8)

    plt.suptitle(f'{btype.title()} ({len(type_pngs)} sprites)', fontsize=14)
    plt.tight_layout()
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [9]:
import shutil

manifest = {}
d = OUTPUT_BASE / 'buildings'
if d.exists():
    keys = sorted(f.stem for f in d.glob('*.png') if not f.stem.startswith('_'))
    if keys:
        manifest['buildings'] = keys

manifest_path = OUTPUT_BASE / 'manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Manifest:')
for cat, keys in manifest.items():
    print(f'  {cat}: {len(keys)}')

zip_path = '/content/buildings_output'
shutil.make_archive(zip_path, 'zip', str(OUTPUT_BASE))
print(f'\nZip: {os.path.getsize(zip_path + ".zip") / 1024 / 1024:.1f} MB')

if ON_COLAB:
    from google.colab import files
    files.download(f'{zip_path}.zip')
else:
    print(f'Download: {zip_path}.zip')

Manifest:
  buildings: 80

Zip: 7.8 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>